In [1]:
import numpy as np

# XOR Dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

def sigmoid(x, derivative=False):
    if derivative:
        return x * (1 - x)
    return 1 / (1 + np.exp(-x))

def initialize_params():
    """Initializes weights with seed 42 to ensure identical results."""
    np.random.seed(42)
    W1 = np.random.randn(2, 4) * 0.5
    b1 = np.zeros((1, 4))
    W2 = np.random.randn(4, 1) * 0.5
    b2 = np.zeros((1, 1))
    return W1, b1, W2, b2

def get_loss_derivative(output, y, loss_type):
    e = 1e-8
    if loss_type == 'mse':
        return (output - y)
    elif loss_type == 'bce':
        return (output - y) / ((output + e) * (1 - output + e))

def train(X, y, lr=0.1, loss_type='mse', epochs=500):
    W1, b1, W2, b2 = initialize_params()
    m = X.shape[0]

    for _ in range(epochs):
        # Forward Pass
        z1 = np.dot(X, W1) + b1
        a1 = sigmoid(z1)
        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)

        # Backward Pass
        dz2 = get_loss_derivative(a2, y, loss_type) * sigmoid(a2, True)
        dW2 = (1/m) * np.dot(a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)

        da1 = np.dot(dz2, W2.T)
        dz1 = da1 * sigmoid(a1, True)
        dW1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)

        # Update Parameters
        W2 -= lr * dW2
        b2 -= lr * db2
        W1 -= lr * dW1
        b1 -= lr * db1
        
    return W1, b1, W2, b2

def get_accuracy(X, y, W1, b1, W2, b2):
    # Final forward pass for prediction
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)
    return np.mean((a2 > 0.5) == y) * 100

# --- Comparison Loops ---

print("--- Learning Rate Comparison ---")
for lr in [0.01, 0.1, 1.0]:
    W1, b1, W2, b2 = train(X, y, lr=lr, loss_type='mse')
    acc = get_accuracy(X, y, W1, b1, W2, b2)
    print(f"LR={lr}, Accuracy={acc:.1f}%")

print("\n--- Loss Function Comparison ---")
for loss in ['mse', 'bce']:
    W1, b1, W2, b2 = train(X, y, lr=0.1, loss_type=loss)
    acc = get_accuracy(X, y, W1, b1, W2, b2)
    print(f"Loss={loss}, Accuracy={acc:.1f}%")

--- Learning Rate Comparison ---
LR=0.01, Accuracy=50.0%
LR=0.1, Accuracy=75.0%
LR=1.0, Accuracy=50.0%

--- Loss Function Comparison ---
Loss=mse, Accuracy=75.0%
Loss=bce, Accuracy=50.0%
